# Drawing Diffusion: an AI diagram-model bake-off, in a notebook

A runnable companion to the ilm.red engineering post
**[Drawing Diffusion](https://ilm.red/blog/drawing-diffusion-ai-diagram-model-bakeoff)**.

The post argues one narrow thing: when a teaching diagram needs **labels**, an image model is the
wrong tool, and a *text* model asked for `SVG` is the right one. This notebook builds that argument
from scratch so you can check it rather than take it on faith:

1. **One word, two pictures** - why grounding on a club's meaning has to happen before anything is drawn.
2. **Scoring a label** - the metric the whole bake-off turns on.
3. **Why raster models misspell** - the failure, measured, on recorded output.
4. **The SVG path** - ask for a drawing, not a painting; the labels are typed characters.
5. **The cost arithmetic** - why ~100x cheaper decides it for a catalogue.

It runs **top to bottom with no API keys and no account**: the raster results are a recorded
fixture captured from the real bake-off, and the SVG engine falls back to a deterministic offline
generator. Drop a real key in the Setup cell to re-run any of it live.

Apache-2.0, like the rest of this repository.

## Setup

Only the standard library is required for the core argument. `cairosvg` is optional - it renders the
SVG we build into a PNG so you can look at it. If it is missing the notebook still runs and simply
skips the picture.

`OPENAI_API_KEY` is optional too. Leave it unset and everything below uses the offline path.

In [ ]:
import os, re, json, math, textwrap
from dataclasses import dataclass

# Optional: only used to rasterise the SVG we generate, purely so you can see it.
try:
    import cairosvg
    HAVE_CAIRO = True
except ImportError:
    try:
        import sys, subprocess
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cairosvg'], check=True)
        import cairosvg
        HAVE_CAIRO = True
    except Exception as e:
        print('cairosvg unavailable (%s) - SVG will be shown as source, not rendered.' % type(e).__name__)
        HAVE_CAIRO = False

# Optional: set this to run the SVG engine against a real model instead of the offline stub.
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')  # e.g. os.environ['OPENAI_API_KEY'] = 'sk-...'
LIVE = bool(OPENAI_API_KEY)
print('mode:', 'LIVE (real model calls)' if LIVE else 'OFFLINE (recorded fixture + deterministic stub)')

## 1. One word, two pictures

Before anything is drawn, something has to decide **what the word means here**. `diffusion` in the
AI Collective is a generative process: start from noise, remove a little at a time, an image appears.
In the physical sciences it is ink spreading through still water. Same six letters, unrelated ideas.

On ilm.red that decision is the Jedi Council's ([previous post](https://ilm.red/blog/meet-the-ai-jedi-council-keeping-a-machine-built-knowledge-graph-honest)).
Here we just hard-code the two senses to make the consequence visible: the *prompt* the drawing engine
receives is different, so the picture is correctly different. Get this step wrong and you ship a
confident illustration of the wrong concept.

In [ ]:
@dataclass
class Sense:
    club: str
    label: str
    gloss: str
    steps: list      # the boxes the diagram must show, in order

AI_SENSE = Sense(
    club='Artificial Intelligence Collective',
    label='Generative Diffusion Models',
    gloss='Start from pure noise and remove a little at a time until a picture appears.',
    steps=['Clean image', 'Add noise', 'Pure noise', 'Denoise step', 'Generated image'])

PHYS_SENSE = Sense(
    club='Physical Sciences',
    label='Molecular Diffusion',
    gloss='Particles drift from crowded regions to empty ones until evenly spread.',
    steps=['Ink drop', 'Concentration gradient', 'Random walk', 'Even distribution'])

def build_prompt(sense: Sense) -> str:
    return textwrap.dedent(f'''
        Draw a labelled teaching diagram of "diffusion".
        Club context: {sense.club}
        Sense: {sense.label} - {sense.gloss}
        The diagram MUST contain exactly these labels, spelled exactly as given:
        {sense.steps}
    ''').strip()

for s in (AI_SENSE, PHYS_SENSE):
    print('=' * 70)
    print(build_prompt(s))

## 2. Scoring a label

The bake-off judges one thing: **are the labels actually right?** So we need a metric before we need
a model.

For each label the diagram was *asked* for, we check whether it appears in the text the diagram
actually *contains*. For a raster image that text comes from OCR. For an SVG it comes from reading
the `<text>` nodes. Same scoring function, two ways of getting the input - which is the whole point.

We report exact-match accuracy, and list what went wrong, because 'four of five labels correct' is
still a broken teaching diagram.

In [ ]:
def score_labels(requested, found):
    """Exact-match label accuracy. `found` is whatever text the diagram really contains."""
    found_norm = {f.strip().lower() for f in found}
    hits, misses = [], []
    for want in requested:
        (hits if want.strip().lower() in found_norm else misses).append(want)
    return {
        'accuracy': len(hits) / len(requested) if requested else 0.0,
        'correct': hits,
        'wrong_or_missing': misses,
        'stray_text': sorted(found_norm - {r.strip().lower() for r in requested}),
    }

def report(engine, requested, found):
    r = score_labels(requested, found)
    print(f"{engine:<28} label accuracy: {r['accuracy']:6.1%}")
    if r['wrong_or_missing']: print(f"{'':<28} missing/misspelled: {r['wrong_or_missing']}")
    if r['stray_text']:       print(f"{'':<28} invented text:      {r['stray_text']}")
    return r

# sanity check the metric itself before trusting it on models
_ = report('self-test (perfect)', ['Add noise', 'Pure noise'], ['Add noise', 'Pure noise'])
_ = report('self-test (broken)',  ['Add noise', 'Pure noise'], ['Add noise', 'Pure nosie', '#F1FC'])

## 3. Why raster models misspell

An image model paints *pixels that look like* text. It has learned the rhythm and shape of writing
without ever learning an alphabet, so a label becomes letter-shaped marks that read as English at a
glance and fall apart on inspection.

Below is what the real bake-off got back, recorded verbatim as an OCR transcript of each generated
image. This is a **fixture**, not a live call - re-running an image model would cost money and would
not reproduce byte-for-byte anyway, which is itself the finding. Set `LIVE` and swap in your own
provider call if you want to check it yourself.

> The relevant literature is in the post's reference list: TextDiffuser (NeurIPS 2023),
> GlyphControl (NeurIPS 2023) and AnyText (ICLR 2024) all exist because this failure is hard.

In [ ]:
# Recorded OCR transcripts from the bake-off images (see the post's figures).
RASTER_FIXTURE = {
    'Qwen-Image': ['Clean image', 'Add nosie', 'Pure noise', 'denohize step by step', '#F1FC', '0.47'],
    'Gemini 2.5 Flash Image': ['Clean image', 'Add noise', 'Pure noise', 'Denoise step', 'Generated imgae'],
    'gpt-image-1': ['Clean image', 'Add noise', 'Pure noise', 'Denoise step', 'Generated image'],
}

requested = AI_SENSE.steps
raster_scores = {}
for engine, ocr_text in RASTER_FIXTURE.items():
    raster_scores[engine] = report(engine, requested, ocr_text)
    print()

print('Note: gpt-image-1 scores 100% *on this run*. Nothing in the method forces it to')
print('do so on the next one - which is the difference between usually and always.')

## 4. The SVG path

So stop asking for a painting and ask for a **drawing**.

A text model is good at structured output. We ask it for `SVG`: a vector diagram whose labels are
real typed characters at real coordinates. The renderer draws the glyphs from a font. There is no
step at which a letter can be hallucinated, because no model is *drawing* letters at all - it is
emitting them as text.

Offline, `svg_engine` returns a deterministic diagram built from the same prompt contract. With a key
set it calls a real model. Either way the next cell scores it the same way we scored the raster output.

In [ ]:
def svg_offline(sense: Sense) -> str:
    """Deterministic stand-in for the text model: same contract, no network."""
    w, gap = 190, 24
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{len(sense.steps)*(w+gap)}" height="200">',
             '<style>text{font-family:Georgia,serif;font-size:15px}</style>']
    for i, step in enumerate(sense.steps):
        x = i * (w + gap)
        parts.append(f'<rect x="{x}" y="60" width="{w}" height="70" rx="12" fill="#f4efe6" stroke="#c0532a"/>')
        parts.append(f'<text x="{x + w//2}" y="100" text-anchor="middle">{step}</text>')
        if i < len(sense.steps) - 1:
            parts.append(f'<line x1="{x+w}" y1="95" x2="{x+w+gap}" y2="95" stroke="#c0532a" stroke-width="2"/>')
    parts.append(f'<text data-role="caption" x="8" y="175" font-size="13" fill="#4a4238">{sense.label} - {sense.club}</text>')
    parts.append('</svg>')
    return ''.join(parts)

def svg_live(sense: Sense) -> str:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'system', 'content': 'You return a single valid <svg> element and nothing else.'},
                  {'role': 'user', 'content': build_prompt(sense)}])
    out = r.choices[0].message.content
    return out[out.index('<svg'): out.rindex('</svg>') + 6]

def svg_engine(sense: Sense) -> str:
    return svg_live(sense) if LIVE else svg_offline(sense)

svg_ai = svg_engine(AI_SENSE)
print(svg_ai[:300], '...')

### Read the labels back out

This is the step that makes the guarantee checkable. We are not trusting the model's word - we parse
the `<text>` nodes out of the SVG it produced and score those. The labels are data, so they can be
verified mechanically before the diagram is ever published.

In [ ]:
def svg_text_nodes(svg: str, include_caption=False):
    """Every label the SVG really contains. The caption is metadata, not a diagram label."""
    out = []
    for m in re.findall(r'<text[^>]*>.*?</text>', svg, re.S):
        if not include_caption and 'data-role="caption"' in m:
            continue
        out.append(re.sub(r'<[^>]+>', '', m))
    return out

found = svg_text_nodes(svg_ai)
print('text nodes found:', found, '\n')
svg_score = report('SVG (text model)', requested, found)

# and the physics sense, from the same engine - the grounding is what differs, not the code
svg_phys = svg_engine(PHYS_SENSE)
print()
_ = report('SVG (text model, physics)', PHYS_SENSE.steps, svg_text_nodes(svg_phys))

if HAVE_CAIRO:
    from IPython.display import Image as IPyImage, display
    for name, svg in (('diffusion - AI sense', svg_ai), ('diffusion - physics sense', svg_phys)):
        print('\n' + name)
        display(IPyImage(cairosvg.svg2png(bytestring=svg.encode(), output_width=900)))

## 5. The cost arithmetic

Correctness settles the argument on its own, but cost is what makes it not even close.

Prices below are per image as published on 3 August 2026 (they move - re-check them). The SVG figure
is the token cost of one small text completion. Scale it across a club catalogue and the difference
stops being a rounding error.

In [ ]:
CATALOGUE = 200   # diagrams in a typical club catalogue

ENGINES = [
    # name,                      usd/image, seconds, guaranteed correct labels?
    ('SVG (text model)',           0.001,     6,     True),
    ('gpt-image-1 (high)',         0.20,     22,     False),
    ('Gemini 2.5 Flash Image',     0.039,    15,     False),
    ('Qwen-Image',                 0.0,      60,     False),
]

print(f"{'engine':<26}{'usd/img':>10}{'catalogue':>12}{'wall clock':>14}  labels")
print('-' * 76)
for name, usd, secs, guaranteed in ENGINES:
    total = usd * CATALOGUE
    hours = secs * CATALOGUE / 3600
    mark = 'guaranteed' if guaranteed else 'best effort'
    print(f'{name:<26}{usd:>10.3f}{total:>11.2f}{hours:>13.1f}h  {mark}')

svg_cost = ENGINES[0][1] * CATALOGUE
gpt_cost = ENGINES[1][1] * CATALOGUE
print(f'\nSVG engine is {gpt_cost / svg_cost:.0f}x cheaper than gpt-image-1 across {CATALOGUE} diagrams,')
print('and it is the only row whose labels are correct by construction rather than by luck.')

## What the bake-off actually decided

Putting the two measurements side by side:

| engine | label accuracy | cost / 200 diagrams | why |
|---|---|---|---|
| **SVG (text model)** | 100%, by construction | ~$0.20 | labels are typed characters, not painted pixels |
| gpt-image-1 | 100% *on this run* | ~$40 | no mechanism forces the next run to be right |
| Gemini 2.5 Flash Image | 80% | ~$7.80 | raster, same class of risk |
| Qwen-Image | 40% + invented text | ~free | misspells and hallucinates stray glyphs |

ilm.red standardised on the **SVG engine** as the default for every term diagram. `gpt-image-1` stays
available for the moments we want a painterly illustration rather than a labelled diagram. Qwen is
retired from this path.

The deeper point is the one from section 1: none of this matters if the *meaning* is wrong. A
perfectly spelled diagram of the wrong sense of a word is still a reader studying the wrong concept.
The drawing engine is the last step, and the cheapest one to get right.

---

**Further reading** - the post's full reference list, with a plain-words page for each idea, is at the
end of [Drawing Diffusion](https://ilm.red/blog/drawing-diffusion-ai-diagram-model-bakeoff).
The concepts here live in the
[Artificial Intelligence Collective](https://ilm.red/book-clubs/ai): 
[diffusion](https://ilm.red/book-clubs/ai/terms/diffusion), 
[text-to-image model](https://ilm.red/book-clubs/ai/terms/text-to-image-model), 
[visual text rendering](https://ilm.red/book-clubs/ai/terms/visual-text-rendering).